In [1]:
from databricks_toolbox.environment.environment_detector import EnvironmentDetector

spark=EnvironmentDetector.get_spark()

2026-08-08 09:30:03,021 - main_logger - INFO - Using standalone Spark (requires Delta storage path)
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/08 09:30:05 WARN Utils: Your hostname, JayStation, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/08 09:30:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/jayboo/projects/sparkstagelapse/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jayboo/.ivy2.5.2/cache
The jars for the packages stored in: /home/jayboo/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c21df8ee-6b04-4d95-9762-59e8abfed2ea;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import random
import string
from datetime import datetime, timedelta

rows = 100

def rand_str(n=5):
    return ''.join(random.choices(string.ascii_letters, k=n))

def rand_date():
    start = datetime(2020, 1, 1)
    return start + timedelta(days=random.randint(0, 2000))

data = [
    (
        i,                                      # id
        rand_str(),                             # name
        random.choice(["Paris","Lyon","Marseille","Toulouse","Bordeaux"]),
        random.choice(["Data engineer","Platform engineer","ML engineer","DevOps engineer"]),
        random.choice([True, False]),           # active
        random.choice(["fr","en","es","de"]),   # lang
        rand_str(3),                            # col7
        random.randint(18, 65),                 # age
        round(random.uniform(30000, 120000),2), # salary
        random.choice(["A","B","C"]),           # grade
        random.choice([True, False]),           # flag1
        random.choice([True, False]),           # flag2
        rand_str(8),                            # misc1
        rand_str(10),                           # misc2
        random.randint(0, 1000),                # score

        # --- added cols ---
        rand_date(),                            # created_at
        rand_date(),                            # updated_at
        random.choice(["IT","HR","Finance","Marketing"]),  # department
        random.choice(["FR","US","DE","ES"]),   # country_code
        random.randint(0, 20),                  # years_exp
        round(random.uniform(0, 5),2),          # rating
        random.choice(["CDI","CDD","Freelance"]), # contract_type
        random.choice([True, False]),           # remote
        rand_str(12),                           # project_code
        random.randint(1000, 9999)              # cost_center
    )
    for i in range(1, rows + 1)
]

columns = [
    "id","name","city","role","active","lang","col7","age","salary","grade",
    "flag1","flag2","misc1","misc2","score",
    "created_at","updated_at","department","country_code","years_exp",
    "rating","contract_type","remote","project_code","cost_center"
]

df = spark.createDataFrame(data, columns)

In [4]:
import pyspark.sql.functions as F

df=df.filter(F.col("active"))
df=df.groupBy("country_code").sum("score")

In [5]:
from sparkstagelapse import display

display(df,n=30,plan=True)

country_code,sum(score)
AllDEESFRUS,All3666432351969914
DE,3666
ES,5196
FR,9914
US,4323
